In [ ]:
-- ============================================
-- Set Role for Liquidity Risk Analysis
-- ============================================
USE ROLE LIQUIDITY_RISK_ROLE;
USE WAREHOUSE LIQUIDITY_RISK_WH;
USE DATABASE LIQUIDITY_RISK_DB;

In [ ]:
from snowflake.snowpark.context import get_active_session
session = get_active_session()
session.query_tag = '{"origin":"sf_sit-is","name":"liquidity_risk_agent","version":{"major":1,"minor":0},"attributes":{"is_quickstart":1,"source":"notebook"}}'

import snowflake.snowpark as snowpark
from snowflake.snowpark import functions as F
from snowflake.snowpark.types import *
import datetime
import sys

import utils
import prod_calculations as pc

In [ ]:
n_days = 150

db = "LIQUIDITY_RISK_DB"
positions_table = f'{db}.RAW.POSITIONS'
inflows_table = f'{db}.RAW.CASH_INFLOWS'
outflows_table = f'{db}.RAW.CASH_OUTFLOWS'
hqla_detail_table = f'{db}.RAW.HQLA_DETAIL'
hqla_summary_table = f'{db}.RAW.HQLA_SUMMARY'
outflows_projections_table = f'{db}.RAW.OUTFLOW_PROJECTIONS'
inflows_projections_table = f'{db}.RAW.INFLOW_PROJECTIONS'
cashflow_projections_table = f'{db}.RAW.NET_CASHFLOW_PROJECTIONS'
lcr_table = f'{db}.PRESENTATION.LCR'

print()
print("Configuration:")
print(f"  DATABASE: {db}")
print()
print("Table names:")
print(f"  POSITIONS_TABLE: {positions_table}")
print(f"  INFLOWS_TABLE: {inflows_table}")
print(f"  OUTFLOWS_TABLE: {outflows_table}")
print(f"  LCR_TABLE: {lcr_table}")

In [ ]:
pc.run_hqla_projection(session, db, n_days, positions_table, hqla_detail_table, 0.9999)
pc.run_summary_hqla(session, n_days, hqla_detail_table, hqla_summary_table)

pc.run_cashflow_projection(session, 'outflow', n_days, outflows_table, outflows_projections_table);
pc.run_cashflow_projection(session, 'inflow', n_days, inflows_table, inflows_projections_table);
pc.run_netting(session, n_days, inflows_projections_table, outflows_projections_table, cashflow_projections_table)

In [ ]:
query = f"""insert into {lcr_table} (DAY_NUMBER, TOTAL_NET_CASH_OUTFLOWS, HQLA, LCR )
select a.day_number,
       a.abs_net_position as total_net_cash_outflows,
       b.total_hqla_usd as hqla,
       hqla / total_net_cash_outflows as lcr
from {cashflow_projections_table} a
join {hqla_summary_table} b on a.day_number = b.day_number
order by day_number
"""

session.sql(query).collect()

In [ ]:
session.sql(f"""
SELECT *
    FROM {lcr_table} 
    WHERE created_timestamp IN (SELECT max(created_timestamp) FROM {lcr_table})
    ORDER BY day_number
"""
).collect()